In [3]:
# Parsing the text labels and verifying the text path

import os
import pandas as pd
from pathlib import Path

# Kaggle input path
BASE_INPUT = Path("/kaggle/input/datasets/nibinv23/iam-handwriting-word-database/iam_words")
WORDS_TXT = BASE_INPUT / "words.txt"
WORDS_IMG_DIR = BASE_INPUT / "words"   

# Below code is used store the id and text in a python list as each tuple has text id and the word.
records = []
with open(WORDS_TXT, "r") as f:
    for line in f:
        line = line.strip() # used to clean a string removes spaces and newline
        if not line or line.startswith("#"):
            continue
        parts = line.split() # breaks the line into list of words using whitespace stores in a list named parts.
        if len(parts) < 9:               # The size of parts array is less than 9 then skip the line.
            continue
        word_id = parts[0]               # e.g., a01-000u-00-00
        seg_status = parts[1]            # ok or err
        transcription = " ".join(parts[8:])  # transcription may contain spaces

        # keep only well‑segmented words
        if seg_status != "ok":
            continue

        records.append((word_id, transcription))

# 2. coonverted into  a DataFrame
df = pd.DataFrame(records, columns=["word_id", "text"])

# 3. Build image paths and verify existence
def build_img_path(word_id):
    # The word_id is like 'a01-000u-00-00'
    # The actual file is under words/a01/a01-000u/a01-000u-00-00.png
    parts = word_id.split("-")
    folder1 = parts[0]                     # "a01"
    folder2 = f"{parts[0]}-{parts[1]}"     # "a01-000u"
    filename = f"{word_id}.png"
    return WORDS_IMG_DIR / folder1 / folder2 / filename

df["img_path"] = df["word_id"].apply(build_img_path)
df["exists"] = df["img_path"].apply(lambda p: p.is_file())

# 4. Filter to only existing images
df_valid = df[df["exists"]].copy()

print(f"Total entries in words.txt (ok only): {len(df)}")
print(f"Images found on disk: {len(df_valid)}")
print(f"Missing images: {len(df) - len(df_valid)}")

# Show a few samples
df.head()

Total entries in words.txt (ok only): 38305
Images found on disk: 38305
Missing images: 0


,word_id,text,img_path,exists
0,a01-000u-00-00,A,/kaggle/input/datasets/nibinv23/iam-handwritin...,True
1,a01-000u-00-01,MOVE,/kaggle/input/datasets/nibinv23/iam-handwritin...,True
2,a01-000u-00-02,to,/kaggle/input/datasets/nibinv23/iam-handwritin...,True
3,a01-000u-00-03,stop,/kaggle/input/datasets/nibinv23/iam-handwritin...,True
4,a01-000u-00-04,Mr.,/kaggle/input/datasets/nibinv23/iam-handwritin...,True


In [3]:
# character set and mapping

from collections import Counter

all_text = ''.join(df_valid['text'].values)
char_counts = Counter(all_text)
chars = sorted(char_counts.keys())

# Add a blank token for CTC (index 0)
chars = ['<blank>'] + chars
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
num_classes = len(chars)

print(f"Number of unique characters: {num_classes}")
print(f"Character set: {chars}")

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# dataset class 
class IAMWordsDataset(Dataset):
    def __init__(self, df, char2idx, target_height=32):
        self.df = df.reset_index(drop=True)
        self.char2idx = char2idx
        self.target_height = target_height
        # Transform: to tensor + standardise pixel values to [0,1]
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Load image as grayscale
        img = Image.open(row['img_path']).convert('L')
        # Resize keeping aspect ratio (height = target_height)
        w, h = img.size
        new_w = int(self.target_height * (w / h))
        img = img.resize((new_w, self.target_height), Image.BICUBIC)
        img_tensor = self.transform(img)  # shape: (1, H, W)
        original_width = img_tensor.shape[2]
        # Encode label
        label = row['text']
        label_enc = [self.char2idx[c] for c in label if c in self.char2idx]
        label_enc = torch.tensor(label_enc, dtype=torch.long)
        # return image tensor, label indices, label string (for evaluation)
        return img_tensor, label_enc, label, original_width

# collate function for various image width
def collate_fn(batch):
    images, labels, texts, widths = zip(*batch)
    # Pad images to the maximum width in the batch
    max_width = max(images, key=lamda x: x.shape[2]).shape[2]
    padded_images = []
    for img in images:
        pad = max_width - img.shape[2]
        # Pad right side with 0 (black)
        padded = torch.nn.functional.pad(img, (0, pad))
        padded_images.append(padded)
    images_batch = torch.stack(padded_images)  # (B, 1, H, W)
    # Labels lengths
    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    # Pad labels
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=0)
    input_lengths = torch.tensor([self._compute_cnn_out_length(w) for w in widths], dtype=torch.long)
    return images_batch, labels_padded, label_lengths, imput_lengths, texts

# dataloaders
dataset = IAMWordsDataset(df_valid, char2idx, target_height=32)
dataloader = DataLoader(dataset, 
                        batch_size=32, 
                        shuffle=True,
                        collate_fn=collate_fn, 
                        num_workers=2
                       )

In [ ]:
# crnn model
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, num_classes, img_height=32, hidden_size=256):
        super(CRNN, self).__init__()
        # CNN feature extractor (output height = 1)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # H/2
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # H/4
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2, 1), (2, 1)),  # H/8, W unchanged
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2, 1), (2, 1)),  # H/16, W unchanged
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2, 1), (2, 1)),  # H/32 → height = 1 if input H=32
        )
        # Map CNN output to sequence
        self.rnn = nn.LSTM(512, hidden_size, num_layers=2,
                           bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # x shape: (B, 1, H, W)
        feats = self.cnn(x)               # (B, 512, 1, W')
        feats = feats.squeeze(2)          # (B, 512, W')
        feats = feats.permute(0, 2, 1)    # (B, W', 512) -> batch_first for RNN
        rnn_out, _ = self.rnn(feats)      # (B, W', hidden*2)
        logits = self.fc(rnn_out)         # (B, W', num_classes)
        # CTC expects log_softmax over class dimension
        return nn.functional.log_softmax(logits, dim=2)